# Clustering — DBSCAN
Algoritmo baseado em densidade: descobre clusters de forma arbitrária e identifica outliers (ruído).

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')

X_scaled = np.load('../dataset/X_scaled.npy')

with open('../dataset/preprocessamento.pkl', 'rb') as f:
    meta = pickle.load(f)

df_orig = meta['df_original']
print(f'Matriz original: {X_scaled.shape}')

## 1. Redução de dimensionalidade com PCA

DBSCAN é baseado em distância. Com muitas dimensões, todas as distâncias ficam parecidas entre si — o algoritmo não consegue distinguir regiões densas de esparsas.  
Isso se chama **maldição da dimensionalidade**: quanto mais colunas, mais o espaço fica "vazio" e o DBSCAN enxerga tudo como ruído ou tudo como um único cluster.

**Solução:** reduzir para um número menor de dimensões com PCA antes de rodar o DBSCAN.

In [ ]:
# Testar quantas componentes capturar
for n in [5, 8, 10, 15, 20]:
    pca_test = PCA(n_components=n, random_state=42)
    pca_test.fit(X_scaled)
    print(f'PCA {n:2d}D → variância explicada: {pca_test.explained_variance_ratio_.sum()*100:.1f}%')

# Usar 10 componentes: bom equilíbrio entre variância (~47%) e dimensionalidade baixa
N_COMPONENTS = 10
pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'\nEspaço escolhido: {N_COMPONENTS}D '
      f'({pca.explained_variance_ratio_.sum()*100:.1f}% da variância original)')

## 2. K-Distance Plot — escolha do eps

Ordena as distâncias ao k-ésimo vizinho mais próximo **no espaço PCA reduzido**.  
O "joelho" da curva indica onde a densidade cai — esse é o `eps` ideal.

In [ ]:
MIN_S = 5  # min_samples (ajuste aqui se necessário)

nbrs = NearestNeighbors(n_neighbors=MIN_S).fit(X_pca)
distancias, _ = nbrs.kneighbors(X_pca)
distancias_k = np.sort(distancias[:, -1])

plt.figure(figsize=(10, 4))
plt.plot(distancias_k, color='steelblue')
plt.title(f'K-Distance Plot (k={MIN_S}) no espaço PCA-{N_COMPONENTS}D — procure o joelho')
plt.xlabel('Pontos ordenados')
plt.ylabel(f'Distância ao {MIN_S}º vizinho')
plt.tight_layout()
plt.show()

print('Percentis das distâncias:')
for p in [50, 75, 85, 90, 95]:
    print(f'  p{p}: {np.percentile(distancias_k, p):.3f}')

## 2. Busca de Parâmetros — grade eps × min_samples
Testa combinações e filtra as que geram pelo menos 2 clusters válidos.

In [ ]:
eps_values  = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]
mins_values = [3, 5, 10, 20]

resultados = []

for eps in eps_values:
    for min_s in mins_values:
        db = DBSCAN(eps=eps, min_samples=min_s)
        lbs = db.fit_predict(X_pca)
        n_clusters = len(set(lbs)) - (1 if -1 in lbs else 0)
        n_noise    = (lbs == -1).sum()
        pct_noise  = n_noise / len(lbs) * 100

        if n_clusters >= 2:
            mask = lbs != -1
            sil = silhouette_score(X_pca[mask], lbs[mask]) if mask.sum() > 1 else -1
            resultados.append({
                'eps': eps, 'min_samples': min_s,
                'n_clusters': n_clusters, 'n_noise': n_noise,
                'pct_noise': round(pct_noise, 1), 'silhouette': round(sil, 4)
            })

df_res = pd.DataFrame(resultados).sort_values('silhouette', ascending=False)
print(f'{len(df_res)} combinações válidas (≥2 clusters):')
display(df_res.head(15))

## 3. Treinar DBSCAN com parâmetros escolhidos
> **Ajuste** `EPS` e `MIN_S` com base na tabela acima.

In [ ]:
EPS   = 2.5  # ajuste com base na tabela acima
MIN_S = 5    # ajuste com base na tabela acima

dbscan = DBSCAN(eps=EPS, min_samples=MIN_S)
labels_db = dbscan.fit_predict(X_pca)

n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise    = (labels_db == -1).sum()

print(f'DBSCAN (eps={EPS}, min_samples={MIN_S}) no espaço PCA-{N_COMPONENTS}D')
print(f'  Clusters encontrados : {n_clusters}')
print(f'  Pontos de ruído      : {n_noise} ({n_noise/len(labels_db)*100:.1f}%)')
print(f'  Distribuição         : {pd.Series(labels_db).value_counts().sort_index().to_dict()}')

mask_valid = labels_db != -1
if mask_valid.sum() > 1 and n_clusters >= 2:
    sil = silhouette_score(X_pca[mask_valid], labels_db[mask_valid])
    dbi = davies_bouldin_score(X_pca[mask_valid], labels_db[mask_valid])
    print(f'  Silhouette (sem ruído): {sil:.4f}')
    print(f'  Davies-Bouldin        : {dbi:.4f}')
else:
    sil, dbi = None, None
    print('Não foi possível calcular métricas. Ajuste EPS e MIN_S acima.')

## 4. Visualização PCA (2D)

In [ ]:
# Projetar o espaço PCA-10D em 2D para visualização
pca2d = PCA(n_components=2, random_state=42)
X_2d = pca2d.fit_transform(X_pca)

palette = plt.cm.tab10.colors
labels_unicas = sorted(set(labels_db))

plt.figure(figsize=(10, 6))
for lbl in labels_unicas:
    mask = labels_db == lbl
    nome = 'Ruído' if lbl == -1 else f'Cluster {lbl}'
    cor  = 'lightgray' if lbl == -1 else palette[lbl % len(palette)]
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=[cor], label=nome, alpha=0.5 if lbl != -1 else 0.3, s=20)

plt.title(f'DBSCAN (eps={EPS}, min_samples={MIN_S}) — PCA {N_COMPONENTS}D → 2D')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Perfil dos clusters (excluindo ruído)

In [6]:
df_perfil = df_orig.copy()
df_perfil['Cluster'] = labels_db

# Apenas pontos não-ruído
df_validos = df_perfil[df_perfil['Cluster'] != -1]

num_cols = ['Age', 'Years_Experience', 'Salary_Before_AI', 'Work_Hours_Per_Week', 'Job_Satisfaction']

print('=== MÉDIAS NUMÉRICAS POR CLUSTER ===')
display(df_validos.groupby('Cluster')[num_cols].mean().round(1))

print('\n=== JOB STATUS POR CLUSTER (%) ===')
status_pct = (df_validos.groupby(['Cluster', 'Job_Status'])
                         .size()
                         .unstack(fill_value=0)
                         .apply(lambda r: r / r.sum() * 100, axis=1)
                         .round(1))
display(status_pct)

=== MÉDIAS NUMÉRICAS POR CLUSTER ===


,Age,Years_Experience,Salary_Before_AI,Work_Hours_Per_Week,Job_Satisfaction
Cluster,,,,,



=== JOB STATUS POR CLUSTER (%) ===


Job_Status
Cluster


## 6. Análise dos outliers (ruído)

In [7]:
df_ruido = df_perfil[df_perfil['Cluster'] == -1]
print(f'Total de outliers: {len(df_ruido)} ({len(df_ruido)/len(df_perfil)*100:.1f}%)')

if len(df_ruido) > 0:
    print('\nDistribuição dos outliers por Job_Status:')
    print(df_ruido['Job_Status'].value_counts())
    print('\nDistribuição por Automation_Risk:')
    print(df_ruido['Automation_Risk'].value_counts())

Total de outliers: 2000 (100.0%)

Distribuição dos outliers por Job_Status:
Job_Status
Unchanged    1093
Modified      801
Replaced      106
Name: count, dtype: int64

Distribuição por Automation_Risk:
Automation_Risk
High      682
Medium    668
Low       650
Name: count, dtype: int64


## 7. Salvar labels para comparação final

In [8]:
import json

np.save('../dataset/labels_dbscan.npy', labels_db)

metricas_db = {
    'silhouette': float(sil) if sil is not None else None,
    'davies_bouldin': float(dbi) if dbi is not None else None,
    'n_clusters': n_clusters,
    'n_noise': int(n_noise),
    'eps': EPS,
    'min_samples': MIN_S
}
with open('../dataset/metricas_dbscan.json', 'w') as f:
    json.dump(metricas_db, f)

print('Labels e métricas salvos.')

Labels e métricas salvos.
